# A scalar autograd engine from scratch (micrograd style)

This notebook builds reverse-mode automatic differentiation for scalars, then a small neural-net library and a training loop on top of it. It matches notes.md section 2 (Backpropagation): the computational graph, the chain rule applied node by node, the `+=` accumulation when a value is used twice, and reverse topological order.

The mechanism in three steps:

1. Every arithmetic op on a `Value` creates a new node that stores its inputs and a function `_backward` that applies the op's local derivative.
2. `loss.backward()` sorts the graph topologically and calls each node's `_backward` in reverse order, so when a node runs, its own `.grad` ($\partial L/\partial \text{node}$) already includes every path from the loss.
3. Each `_backward` applies one instance of the chain rule, `child.grad += local_derivative * node.grad`. The `+=` is required: if a value is used in two places, the gradients from both paths add (multivariate chain rule).

Demos, in order: the `+=` rule on $L = a \cdot a$, the hand-worked network from notes section 2 checked number by number, a gradient check of every op against finite differences, and training a 2-16-16-1 MLP on a small 2-class dataset. Only numpy is needed (for data and random init). The training cell takes one to two minutes on a laptop CPU; the rest is instant. Run it with
`uv run jupyter execute ml/03_neural_networks/micrograd_plus.ipynb`.

In [1]:
import math

import numpy as np

## The `Value` class

A `Value` holds a float (`data`), its gradient (`grad`, starting at 0), the `Value`s it was computed from (`_prev`), and the `_backward` function. The ops implemented directly are `+`, `*`, `**` (constant exponent), `tanh`, `relu`, `exp` and `log`; subtraction, negation and division are built from these. The three backward rules from notes section 2 appear here: addition passes the gradient unchanged to both inputs, multiplication scales each input's gradient by the other input's forward value, and a unary op scales by its derivative at the forward value.

The next cell defines the class and runs the smallest case of the `+=` rule (Exercise 3): $L = a \cdot a$ with $a = 3$. Expected: `a.grad = 6`, the sum of two contributions of 3, one per use of $a$. With `=` instead of `+=` the second contribution would overwrite the first and give 3.

In [2]:
class Value:
    # A scalar with reverse-mode autodiff.
    #   data:      forward value (float)
    #   grad:      dLoss/dThis, filled in by .backward(); starts at 0
    #   _prev:     the Values this one was computed from (graph edges)
    #   _backward: function that adds this node's contribution to its inputs' grads
    #   _op:       op name, for printing only

    def __init__(self, data, _children=(), _op=""):
        self.data = float(data)
        self.grad = 0.0
        self._prev = set(_children)
        self._backward = lambda: None  # leaves have no inputs
        self._op = _op

    # ---- binary ops --------------------------------------------------------

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            # c = a + b  =>  dc/da = 1, dc/db = 1: the gradient at c is passed
            # unchanged to both a and b (changing a by h changes c by h).
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad

        out._backward = _backward
        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            # c = a * b  =>  dc/da = b, dc/db = a ("swap and multiply").
            # The local gradient is the other input's FORWARD value, which is
            # why backward needs the forward values kept in memory.
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad

        out._backward = _backward
        return out

    def __pow__(self, k):
        # Constant (non-Value) exponents only: covers x**2 and x**-1 (division).
        assert isinstance(k, (int, float)), "only supports int/float powers"
        out = Value(self.data ** k, (self,), f"**{k}")

        def _backward():
            # c = a^k  =>  dc/da = k * a^(k-1) (power rule)
            self.grad += (k * self.data ** (k - 1)) * out.grad

        out._backward = _backward
        return out

    # ---- unary nonlinearities ---------------------------------------------

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            # d/dz tanh(z) = 1 - tanh(z)^2, computed from the stored OUTPUT t.
            # Since t is in (-1, 1), the factor is in (0, 1]: every tanh shrinks
            # the gradient, and near saturation (|t| close to 1) the factor is
            # close to 0.
            self.grad += (1.0 - t * t) * out.grad

        out._backward = _backward
        return out

    def relu(self):
        out = Value(self.data if self.data > 0 else 0.0, (self,), "relu")

        def _backward():
            # derivative 1 where the unit was active, 0 where it was off, so the
            # gradient passes unchanged or not at all (subgradient 0 at z = 0)
            self.grad += (1.0 if out.data > 0 else 0.0) * out.grad

        out._backward = _backward
        return out

    def exp(self):
        e = math.exp(self.data)
        out = Value(e, (self,), "exp")

        def _backward():
            # d/dz e^z = e^z, so the local gradient is the forward output
            self.grad += e * out.grad

        out._backward = _backward
        return out

    def log(self):
        assert self.data > 0, "log needs a positive input"
        out = Value(math.log(self.data), (self,), "log")

        def _backward():
            # d/dz log(z) = 1/z, which grows without bound as z -> 0+. This is why
            # cross-entropy has a large gradient on confidently wrong predictions
            # and why computing log(softmax) naively is numerically fragile.
            self.grad += (1.0 / self.data) * out.grad

        out._backward = _backward
        return out

    # ---- derived ops: everything below reduces to +, *, ** ------------------

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __truediv__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return self * other ** -1  # a / b = a * b^-1

    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __rsub__(self, other):
        return (-self) + other

    def __rtruediv__(self, other):
        return Value(other) * self ** -1

    # ---- the backward pass -------------------------------------------------

    def backward(self):
        # A node's _backward multiplies by out.grad, so out.grad must already
        # hold the full dLoss/dOut, i.e. every path from the loss to `out` must
        # have been processed. Reverse topological order guarantees this and
        # visits each node once, so backprop costs O(graph size) rather than
        # O(number of paths), which can be exponential in depth.
        topo, visited = [], set()

        def build(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build(child)
                topo.append(v)  # post-order: inputs always come before outputs

        build(self)

        self.grad = 1.0  # seed: dLoss/dLoss = 1
        for node in reversed(topo):
            node._backward()

    def __repr__(self):
        return f"Value(data={self.data:.6f}, grad={self.grad:.6f})"


# Demo: a value used twice. L = a * a, so dL/da = 2a = 6 at a = 3.
a = Value(3.0)
L = a * a
L.backward()
print(f"L = a*a at a=3: L = {L.data}, a.grad = {a.grad}  (expected 2a = 6)")
assert a.grad == 6.0

L = a*a at a=3: L = 9.0, a.grad = 6.0  (expected 2a = 6)


## The hand-worked network from notes section 2

Notes section 2 backpropagates by hand through $z_1 = w_1 x + b_1$, $h = \tanh(z_1)$, $\hat y = w_2 h + b_2$, $L = (\hat y - y)^2$ with $x = 1$, $y = 1$, $w_1 = 0.5$, $b_1 = 0.1$, $w_2 = -0.3$, $b_2 = 0.2$. The next cell builds the same graph with `Value`, runs `backward()`, and compares each gradient with the value in the notes. It then takes one gradient-descent step with $\eta = 0.1$ and recomputes the loss. Expected: $L = 0.923742$, gradients $\partial L/\partial w_2 = -1.032333$, $\partial L/\partial b_2 = -1.922230$, $\partial L/\partial w_1 = \partial L/\partial b_1 = 0.410345$, and a loss of about 0.49 after the step.

In [3]:
def tiny_net(w1, b1, w2, b2, x=1.0, y=1.0):
    h = (w1 * x + b1).tanh()
    y_hat = w2 * h + b2
    return (y_hat - y) ** 2


params = {"w1": Value(0.5), "b1": Value(0.1), "w2": Value(-0.3), "b2": Value(0.2)}
L = tiny_net(**params)
L.backward()

notes_grads = {"w1": 0.410345, "b1": 0.410345, "w2": -1.032333, "b2": -1.922230}
print(f"L = {L.data:.6f}  (notes: 0.923742)")
for name, p in params.items():
    print(f"  dL/d{name} = {p.grad:+.6f}   (notes: {notes_grads[name]:+.6f})")
    assert abs(p.grad - notes_grads[name]) < 1e-6

eta = 0.1
stepped = {name: Value(p.data - eta * p.grad) for name, p in params.items()}
print(f"loss after one step with eta = {eta}: {tiny_net(**stepped).data:.4f}  (notes: about 0.49)")

L = 0.923742  (notes: 0.923742)
  dL/dw1 = +0.410345   (notes: +0.410345)
  dL/db1 = +0.410345   (notes: +0.410345)
  dL/dw2 = -1.032333   (notes: -1.032333)
  dL/db2 = -1.922230   (notes: -1.922230)
loss after one step with eta = 0.1: 0.4920  (notes: about 0.49)


## Gradient check: every op against finite differences

To check the engine, the next cell builds one scalar expression that uses every op (`+ - * / ** tanh relu exp log`) and reuses variables on several paths, so the `+=` accumulation is also tested. It compares each leaf's backprop gradient with the centered finite difference $(f(x+h) - f(x-h))/(2h)$, whose error is $O(h^2)$ because the odd terms of the Taylor expansion cancel (a one-sided difference has $O(h)$ error). Expected: relative errors around $10^{-10}$ in float64; the cell asserts the maximum is below $10^{-6}$.

In [4]:
def expr(a, b, c, d):
    e = (a * b + c).tanh()
    f = (a - d).relu() + (b * c).exp() * 0.1
    g = (e * f + a / d) ** 2
    h = (f + 2.0).log() + g
    return h * c + e


def gradient_check():
    leaf_vals = {"a": 0.7, "b": -0.4, "c": 1.3, "d": 2.1}

    # analytic gradients from the engine
    leaves = {k: Value(v) for k, v in leaf_vals.items()}
    expr(**leaves).backward()

    # numeric gradients from centered differences
    h = 1e-6
    print(f"{'leaf':>4} | {'analytic (backprop)':>20} | {'numeric (fin diff)':>20} | {'rel err':>10}")
    max_rel = 0.0
    for name in leaf_vals:
        vals_p, vals_m = dict(leaf_vals), dict(leaf_vals)
        vals_p[name] += h
        vals_m[name] -= h
        fp = expr(**{k: Value(v) for k, v in vals_p.items()}).data
        fm = expr(**{k: Value(v) for k, v in vals_m.items()}).data
        numeric = (fp - fm) / (2 * h)
        analytic = leaves[name].grad
        rel = abs(analytic - numeric) / (abs(analytic) + abs(numeric) + 1e-12)
        max_rel = max(max_rel, rel)
        print(f"{name:>4} | {analytic:>20.10f} | {numeric:>20.10f} | {rel:>10.2e}")

    assert max_rel < 1e-6, f"gradient check FAILED (max rel err {max_rel:.2e})"
    print(f"max relative error: {max_rel:.2e}  -> PASS")


gradient_check()

leaf |  analytic (backprop) |   numeric (fin diff) |    rel err
   a |         0.2968967581 |         0.2968967580 |   1.76e-10
   b |         0.4092567816 |         0.4092567816 |   9.08e-11
   c |         1.2642760496 |         1.2642760496 |   6.30e-12
   d |        -0.1564554048 |        -0.1564554049 |   2.37e-10
max relative error: 2.37e-10  -> PASS


## A small neural-net library on top of `Value`

`Neuron` computes $\tanh(w \cdot x + b)$ (or just $w \cdot x + b$ for the output layer), `Layer` is a list of neurons, and `MLP` chains layers. Weights are drawn from $U[-1, 1]$ scaled by $1/\sqrt{n_{in}}$, following the variance argument of notes section 5: $\mathrm{Var}(w \cdot x) = n_{in}\,\mathrm{Var}(w)\,\mathrm{Var}(x)$, so $\mathrm{Var}(w)$ must scale like $1/n_{in}$ to keep activations from growing or shrinking with width. The last layer is linear because it outputs a score; the loss decides how the score is turned into a prediction.

The next cell defines the classes and builds the 2-16-16-1 network used below. Expected: $2 \cdot 16 + 16 + 16 \cdot 16 + 16 + 16 + 1 = 337$ scalar parameters.

In [5]:
class Neuron:
    def __init__(self, n_in, nonlin=True):
        scale = 1.0 / math.sqrt(n_in)
        self.w = [Value(np.random.uniform(-1, 1) * scale) for _ in range(n_in)]
        self.b = Value(0.0)
        self.nonlin = nonlin

    def __call__(self, x):
        # sum() with a Value start value builds the graph w1*x1 + ... + wn*xn + b
        act = sum((wi * xi for wi, xi in zip(self.w, x)), self.b)
        return act.tanh() if self.nonlin else act

    def parameters(self):
        return self.w + [self.b]


class Layer:
    def __init__(self, n_in, n_out, nonlin=True):
        self.neurons = [Neuron(n_in, nonlin) for _ in range(n_out)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]


class MLP:
    # Multi-layer perceptron; the last layer is linear (it outputs a score).

    def __init__(self, n_in, layer_sizes):
        sizes = [n_in] + layer_sizes
        self.layers = [
            Layer(sizes[i], sizes[i + 1], nonlin=(i != len(layer_sizes) - 1))
            for i in range(len(layer_sizes))
        ]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]


print("parameters in a 2-16-16-1 MLP:", len(MLP(2, [16, 16, 1]).parameters()))

parameters in a 2-16-16-1 MLP: 337


## Training the MLP on a 2-class dataset

The data are two interleaving noisy arcs ("two moons"), 30 points per class with labels $+1$ and $-1$; no straight line separates them, so the hidden layers are needed. The loss is the squared hinge loss $\max(0, 1 - y \cdot \text{score})^2$, averaged over the data, plus a small L2 penalty $10^{-4}\sum w^2$. Each step builds one graph over the whole dataset, zeroes all gradients (because `backward()` accumulates with `+=`), runs `backward()`, and updates every parameter with $p \leftarrow p - \eta\,\partial L/\partial p$. The learning rate decays linearly from 0.5 to 0.05 over 300 steps.

Expected: the loss falls from about 1.0 to about 0.003, and training accuracy goes from about 57% to 100% (reached by step 90). The cell asserts a final accuracy of at least 95%.

In [6]:
def make_moons_ish(n_per_class=30):
    t = np.random.uniform(0, math.pi, n_per_class)
    x0 = np.stack([np.cos(t), np.sin(t)], axis=1) + np.random.randn(n_per_class, 2) * 0.1
    x1 = np.stack([1 - np.cos(t), -np.sin(t) + 0.3], axis=1) + np.random.randn(n_per_class, 2) * 0.1
    X = np.concatenate([x0, x1])
    y = np.array([1.0] * n_per_class + [-1.0] * n_per_class)
    return X, y


def train_mlp(n_steps=300):
    np.random.seed(42)
    X, y = make_moons_ish(30)
    model = MLP(2, [16, 16, 1])
    print(f"dataset: {len(X)} points, model: {len(model.parameters())} scalar parameters")

    for step in range(n_steps + 1):
        lr = 0.5 * (1.0 - 0.9 * step / n_steps)  # linear decay from 0.5 to 0.05

        # forward: one graph over the whole dataset
        scores = [model([Value(xi) for xi in xrow]) for xrow in X]
        losses = [(1.0 + (-yi) * si).relu() ** 2 for si, yi in zip(scores, y)]  # squared hinge
        data_loss = sum(losses, Value(0.0)) * (1.0 / len(losses))
        reg_loss = sum((p * p for p in model.parameters()), Value(0.0)) * 1e-4
        loss = data_loss + reg_loss

        # backward: zero first, since backward() accumulates with +=
        for p in model.parameters():
            p.grad = 0.0
        loss.backward()

        # SGD update: step against the gradient
        for p in model.parameters():
            p.data -= lr * p.grad

        if step % 30 == 0:
            acc = np.mean([(s.data > 0) == (yi > 0) for s, yi in zip(scores, y)])
            print(f"step {step:3d} | lr {lr:.4f} | loss {loss.data:.4f} | accuracy {acc * 100:5.1f}%")

    final_acc = np.mean([(model([Value(xi) for xi in xrow]).data > 0) == (yi > 0)
                         for xrow, yi in zip(X, y)])
    print(f"final training accuracy: {final_acc * 100:.1f}%")
    return final_acc


final_acc = train_mlp()
assert final_acc >= 0.95, "training failed to fit the tiny dataset"

dataset: 60 points, model: 337 scalar parameters
step   0 | lr 0.5000 | loss 0.9928 | accuracy  56.7%


step  30 | lr 0.4550 | loss 0.1852 | accuracy  93.3%


step  60 | lr 0.4100 | loss 0.0709 | accuracy  98.3%


step  90 | lr 0.3650 | loss 0.0117 | accuracy 100.0%


step 120 | lr 0.3200 | loss 0.0048 | accuracy 100.0%


...

step 180 | lr 0.2300 | loss 0.0031 | accuracy 100.0%


step 210 | lr 0.1850 | loss 0.0030 | accuracy 100.0%


step 240 | lr 0.1400 | loss 0.0029 | accuracy 100.0%


step 270 | lr 0.0950 | loss 0.0029 | accuracy 100.0%


step 300 | lr 0.0500 | loss 0.0029 | accuracy 100.0%
final training accuracy: 100.0%


**Reading the output.** Accuracy at step 0 is near chance, the loss drops quickly in the first 90 steps while the learning rate is large, and the remaining steps reduce the loss further at a smaller learning rate. Every gradient used here came from the same `+=` chain-rule mechanism checked above; the engine has no other source of derivatives.